In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
import pickle

# Load the dataset
df = pd.read_csv('vegemite.csv')

print(f"Original dataset shape: {df.shape}")

# Step 1: Data Preparation
# 1. Shuffle the dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 2. Randomly take out 1000 data points with balanced classes
test_samples = []
class_distribution = {0: 334, 1: 333, 2: 333}

for class_value, sample_count in class_distribution.items():
    class_rows = df[df['Class'] == class_value]

    if len(class_rows) < sample_count:
        print(f"Warning: Not enough samples for class {class_value}. Only {len(class_rows)} available.")
        sample_count = len(class_rows)

    selected = class_rows.sample(n=sample_count, random_state=42)
    test_samples.append(selected)
    print(f"Selected {sample_count} samples from class {class_value}")

    # Remove selected samples from original dataframe
    df = df[~df.index.isin(selected.index)]

test_df = pd.concat(test_samples).reset_index(drop=True)
train_df = df.reset_index(drop=True)

print(f"Test set shape: {test_df.shape}")
print(f"Training set shape: {train_df.shape}")

# Check for constant columns
constant_cols = [col for col in train_df.columns if train_df[col].nunique() == 1]

if constant_cols:
    # Remove constant columns
    train_df = train_df.drop(columns=constant_cols)
    test_df = test_df.drop(columns=constant_cols)
    print(f"Removed {len(constant_cols)} constant columns: {constant_cols}")

# Check for columns with few integer values
few_integer_cols = []
for col in train_df.columns:
    if train_df[col].dtype in ['int64', 'float64']:
        unique_values = train_df[col].nunique()
        if unique_values < 10 and col != 'Class':
            few_integer_cols.append(col)

# Convert integer columns with few values to category
for col in few_integer_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')
    print(f"Converted {col} to categorical")

# Check class distribution
class_counts = train_df['Class'].value_counts()
print("Class distribution before resampling:")
print(class_counts)

# Calculate imbalance ratio
min_class_count = class_counts.min()
max_class_count = class_counts.max()
imbalance_ratio = min_class_count / max_class_count

print(f"Imbalance ratio (min/max): {imbalance_ratio:.4f}")

# Define thresholds for resampling
BALANCED_THRESHOLD = 0.8
MODERATE_IMBALANCE = 0.5
SEVERE_IMBALANCE = 0.2

# Decide on resampling method
if imbalance_ratio >= BALANCED_THRESHOLD:
    print("Classes are fairly balanced. No resampling needed.")
    resampling_method = None
elif imbalance_ratio >= MODERATE_IMBALANCE:
    print("Moderate class imbalance detected. Will use class weights in model training.")
    class_weights = {i: len(train_df) / (len(np.unique(train_df['Class'])) * count) 
                     for i, count in enumerate(class_counts)}
    print(f"Class weights: {class_weights}")
    resampling_method = "class_weights"
elif imbalance_ratio >= SEVERE_IMBALANCE:
    print("Significant class imbalance detected. Will apply SMOTE oversampling.")
    resampling_method = "smote"
else:
    print("Severe class imbalance detected. Will apply combination of undersampling and SMOTE.")
    resampling_method = "undersampling+smote"

# Apply resampling if needed
X = train_df.drop('Class', axis=1)
y = train_df['Class']

if resampling_method in ["smote", "undersampling+smote"]:
    # Apply SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y)
    
    # Create a new DataFrame with the resampled data
    resampled_df = pd.DataFrame(X_resampled, columns=X.columns)
    resampled_df['Class'] = y_resampled
    
    print("Class distribution after resampling:")
    print(pd.Series(y_resampled).value_counts())
    
    # Replace the original training data with the resampled data
    train_df = resampled_df
else:
    print("No resampling applied.")

# Create composite features
# Average temperature 
train_df['FFTE_avg_temperature'] = (train_df['FFTE Temperature 1 - 1'] + 
                                   train_df['FFTE Temperature 1 - 2'] +
                                   train_df['FFTE Temperature 2 - 1'] +
                                   train_df['FFTE Temperature 3 - 2']) / 4

# Ratio between feed flow and steam pressure setpoints  
train_df['flow_pressure_ratio'] = train_df['FFTE Feed flow SP'] / (train_df['FFTE Steam pressure SP'] + 1e-10)  # Adding small value to avoid division by zero

# Deviation between actual and target solids 
train_df['FFTE_Production_solids_deviation'] = train_df['FFTE Production solids PV'] - train_df['FFTE Production solids SP']

# Add the same composite features to test_df
test_df['FFTE_avg_temperature'] = (test_df['FFTE Temperature 1 - 1'] + 
                                 test_df['FFTE Temperature 1 - 2'] +
                                 test_df['FFTE Temperature 2 - 1'] +
                                 test_df['FFTE Temperature 3 - 2']) / 4

test_df['flow_pressure_ratio'] = test_df['FFTE Feed flow SP'] / (test_df['FFTE Steam pressure SP'] + 1e-10)
test_df['FFTE_Production_solids_deviation'] = test_df['FFTE Production solids PV'] - test_df['FFTE Production solids SP']

print(f"Number of features in final dataset: {train_df.shape[1] - 1}")  # Subtract 1 for the Class column

# Step 2: Feature selection, Model Training and Evaluation
X_train = train_df.drop("Class", axis=1)
y_train = train_df["Class"]
X_test = test_df.drop("Class", axis=1)
y_test = test_df["Class"]

# Handle categorical features
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Ensure X_train and X_test have the same columns
missing_cols = set(X_train.columns) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = 0

# Ensure the columns are in the same order
X_test = X_test[X_train.columns]

# Feature selection
selector = SelectKBest(f_classif, k='all')
selector.fit(X_train, y_train)

# Create DataFrame with feature names and scores
feature_scores = pd.DataFrame({
    'Feature': X_train.columns,
    'Score': selector.scores_
})

# Sort features by importance
feature_scores = feature_scores.sort_values('Score', ascending=False)
print("Top 10 features by importance:")
print(feature_scores.head(10))

# Select top 20 features (or adjust as needed)
top_features = feature_scores.head(20)['Feature'].values
X_train_selected = X_train[top_features]
X_test_selected = X_test[top_features]

print(f"Selected {len(top_features)} features for model training")

# Model training and evaluation
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, multi_class='multinomial')
}

results = {}
model_performances = {}

print("\nModel Evaluation on Test Dataset:")
print("-" * 50)

for name, model in models.items():
    # Train the model
    model.fit(X_train_selected, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_selected)
    
    # Evaluate model
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_test, y_pred)
    
    # Store results
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'report': report,
        'confusion_matrix': conf_matrix
    }
    
    # Store performance metrics for comparison
    model_performances[name] = {
        'Accuracy': accuracy,
        'Precision (macro avg)': report['macro avg']['precision'],
        'Recall (macro avg)': report['macro avg']['recall'],
        'F1-score (macro avg)': report['macro avg']['f1-score']
    }
    
    print(f"\nModel: {name}")
    print(f"Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(conf_matrix)

# Create a comparison table
comparison_df = pd.DataFrame(model_performances).T
print("\nModel Comparison:")
print(comparison_df)

# Find the best model
best_model_name = comparison_df['F1-score (macro avg)'].idxmax()
print(f"\nBest performing model: {best_model_name} with F1-score: {comparison_df.loc[best_model_name, 'F1-score (macro avg)']:.4f}")

# Save the best model
best_model = results[best_model_name]['model']
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print(f"Saved best model ({best_model_name}) to 'best_model.pkl'")

# Step 3: ML to AI
# Load the saved model
with open('best_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Predict using the test dataset
test_predictions = loaded_model.predict(X_test_selected)
test_accuracy = accuracy_score(y_test, test_predictions)
test_report = classification_report(y_test, test_predictions, output_dict=True)

print("\nStep 3: ML to AI - Testing model on 1000 unseen data points")
print(f"Loaded model accuracy: {test_accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, test_predictions))

# Test all models on the test dataset
all_model_test_performances = {}
for name, model_info in results.items():
    model = model_info['model']
    test_pred = model.predict(X_test_selected)
    test_acc = accuracy_score(y_test, test_pred)
    test_rep = classification_report(y_test, test_pred, output_dict=True)
    
    all_model_test_performances[name] = {
        'Test Accuracy': test_acc,
        'Test Precision': test_rep['macro avg']['precision'],
        'Test Recall': test_rep['macro avg']['recall'],
        'Test F1-score': test_rep['macro avg']['f1-score']
    }

# Create comparison table for test results
test_comparison_df = pd.DataFrame(all_model_test_performances).T
print("\nModel Comparison on Test Dataset:")
print(test_comparison_df)

# Check if best model on training data is also best on test data
test_best_model = test_comparison_df['Test F1-score'].idxmax()
print(f"Best model on test data: {test_best_model}")
print(f"Is it the same as selected best model? {'Yes' if test_best_model == best_model_name else 'No'}")

# Step 4: Develop rules from the ML model
# Filter features to only include SP (set point) features
sp_features = [col for col in train_df.columns if col.endswith('SP') and col != 'Class']
print(f"\nStep 4: Developing rules using only SP features ({len(sp_features)} features)")

# Prepare data with only SP features
X_sp = train_df[sp_features]
y_sp = train_df['Class']

# One-hot encode categorical features if any
X_sp_encoded = pd.get_dummies(X_sp, drop_first=True)

# Train decision tree model
dt_rule_model = DecisionTreeClassifier(max_depth=5, random_state=42)  # Limiting depth for interpretability
dt_rule_model.fit(X_sp_encoded, y_sp)

# Print the decision tree
tree_rules = export_text(dt_rule_model, feature_names=list(X_sp_encoded.columns))
print("\nDecision Tree Rules:")
print(tree_rules)

# Extract rules in a more readable format
def extract_rules(tree, feature_names, class_names=None):
    tree_ = tree.tree_
    feature_name = [
        feature_names[i] if i != -2 else "undefined!"
        for i in tree_.feature
    ]
    
    paths = []
    path = []
    
    def dfs(node, path, paths):
        if tree_.feature[node] != -2:  # internal node
            name = feature_name[node]
            threshold = tree_.threshold[node]
            
            # Left child: feature <= threshold
            path.append((name, "<=", threshold))
            dfs(tree_.children_left[node], path, paths)
            path.pop()
            
            # Right child: feature > threshold
            path.append((name, ">", threshold))
            dfs(tree_.children_right[node], path, paths)
            path.pop()
        else:  # leaf node
            class_counts = tree_.value[node][0]
            class_index = np.argmax(class_counts)
            class_label = class_names[class_index] if class_names else class_index
            paths.append((path.copy(), class_label, class_counts))
    
    dfs(0, path, paths)
    
    # Convert paths to rules
    rules = []
    for path, class_label, class_counts in paths:
        if path:  # Skip empty paths
            rule = "For class " + str(class_label) + ": "
            for i, (feature, op, threshold) in enumerate(path):
                if i > 0:
                    rule += " AND "
                rule += f"{feature} {op} {threshold:.2f}"
            
            # Add confidence information
            total = sum(class_counts)
            confidence = (class_counts[class_label] / total) * 100 if total > 0 else 0
            rule += f" (Confidence: {confidence:.1f}%)"
            
            rules.append(rule)
    
    return rules

# Extract and print readable rules
class_names = [0, 1, 2]  # Class labels
rules = extract_rules(dt_rule_model, feature_names=list(X_sp_encoded.columns), class_names=class_names)

print("\nExtracted Rules for Vegemite Production:")
for rule in rules:
    print(rule)

print("\nAssignment Complete!")

Original dataset shape: (15237, 47)
Selected 334 samples from class 0
Selected 333 samples from class 1
Selected 333 samples from class 2
Test set shape: (1000, 47)
Training set shape: (14237, 47)
Removed 2 constant columns: ['TFE Steam temperature SP', 'TFE Product out temperature']
Converted FFTE Feed tank level SP to categorical
Converted FFTE Pump 1 to categorical
Converted FFTE Pump 1 - 2 to categorical
Converted FFTE Pump 2 to categorical
Converted TFE Motor speed to categorical
Class distribution before resampling:
Class
2    7215
1    4714
0    2308
Name: count, dtype: int64
Imbalance ratio (min/max): 0.3199
Significant class imbalance detected. Will apply SMOTE oversampling.
Class distribution after resampling:
Class
2    7215
0    7215
1    7215
Name: count, dtype: int64
Number of features in final dataset: 47
Top 10 features by importance:
                      Feature       Score
20     FFTE Temperature 1 - 1  507.177963
25     FFTE Temperature 3 - 2  506.397598
2          

/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Model: Logistic Regression
Accuracy: 0.5060
Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.63      0.55       334
           1       0.45      0.27      0.34       333
           2       0.55      0.62      0.58       333

    accuracy                           0.51      1000
   macro avg       0.50      0.51      0.49      1000
weighted avg       0.50      0.51      0.49      1000

Confusion Matrix:
[[209  55  70]
 [145  91  97]
 [ 69  58 206]]

Model Comparison:
                     Accuracy  Precision (macro avg)  Recall (macro avg)  \
Decision Tree           0.979               0.979070            0.979009   
Random Forest           0.995               0.995010            0.995001   
Gradient Boosting       0.890               0.889810            0.889959   
SVM                     0.475               0.577924            0.474672   
KNN                     0.939               0.938996            0.938990   
Logistic Reg